# Notebook 03: Continuous HMM (Baum-Welch, No Jumps)

This notebook demonstrates the **Continuous Gaussian Hidden Markov Model** (CHMM), the central
methodological contribution of this project. Unlike the discrete HMM baseline from
[JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl) — which bins observations into categorical
levels and counts transition frequencies — the CHMM models each hidden state's emission as a
Gaussian distribution $\mathcal{N}(\mu_k, \sigma_k)$ and learns **all** parameters (transition
matrix, emission means, emission standard deviations) directly from raw continuous returns via the
**Baum-Welch** (Expectation-Maximization) algorithm. This eliminates quantization error and provides
a principled probabilistic framework.

**No jump mechanisms are used in this notebook.** The jump-augmented model is explored separately.

### Sections
1. Setup
2. Configuration (tunable hyperparameters)
3. Load data and compute returns
4. Train model via Baum-Welch
5. Convergence diagnostics
6. Emission distribution visualization
7. Transition matrix heatmap
8. Stationary distribution
9. Residence times
10. Viterbi decoding and regime overlay
11. Simulate N_PATHS Monte Carlo paths
12. Validation metrics (KS, kurtosis, ACF-MAE, Wasserstein, Hellinger)
13. Figure 3: three-panel in-sample comparison
14. Disclaimer

## Setup

In [ ]:
include("../Include.jl");

## Configuration

Tune these parameters interactively. `K` controls model complexity (number of hidden regimes),
`MAX_ITER` caps the EM iterations, and `N_PATHS` sets the number of Monte Carlo simulation paths
used for validation.

In [ ]:
# --- TUNABLE CONFIGURATION ---
ticker = "SPY";              # asset to model
K = 6;                        # number of hidden states (try 4, 6, 8, 10, 13)
MAX_ITER = 60;                # maximum Baum-Welch iterations
N_PATHS = 1000;               # number of Monte Carlo simulation paths
L = 252;                      # ACF max lag (1 trading year)
risk_free_rate = 0.0;         # annualized risk-free rate
Δt = 1/252;                   # daily time step in years

## Load Data and Compute Returns

We load the in-sample S&P 500 OHLCV data (2014--2024) and compute annualized excess log returns:
$$G_t = \frac{1}{\Delta t} \ln\!\left(\frac{P_t}{P_{t-1}}\right) - r_f$$
with $\Delta t = 1/252$ (daily) and $r_f = 0$ by default.

In [ ]:
# --- LOAD IN-SAMPLE DATA (2014-2024) ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

# Filter to tickers with complete history
dataset = Dict{String,DataFrame}();
for (t, data) ∈ train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

# Compute excess log growth rates for the target ticker
R = log_growth_matrix(dataset, ticker; Δt=Δt, risk_free_rate=risk_free_rate);

println("Loaded $(ticker): T = $(length(R)) daily observations");
println("Mean = $(round(mean(R), digits=2)), Std = $(round(std(R), digits=2))");

## Train Continuous HMM via Baum-Welch

The `build` factory calls `baum_welch()` internally. Initialization uses **quantile-based chunking**:
observations are sorted and split into $K$ equal-sized segments, whose sample mean and standard
deviation seed the emission parameters. This prevents degenerate solutions that plague random
initialization. The model is callable as a functor: `model(start_state, n_steps)` returns a
state chain.

In [ ]:
# --- TRAIN MODEL ---
model = build(MyContinuousHiddenMarkovModel, (
    observations = R,
    number_of_states = K,
    max_iter = MAX_ITER
));

println("Training complete: $(length(model.log_likelihood_history)) EM iterations");
println("Final log-likelihood: $(round(model.log_likelihood_history[end], digits=2))");

## Convergence Plot

The log-likelihood should increase monotonically at each EM step. A plateau indicates convergence.
The internal tolerance is `1e-4`.

In [ ]:
# --- CONVERGENCE PLOT ---
p_conv = plot(model.log_likelihood_history,
    title="Baum-Welch Convergence ($(ticker), K=$(K))",
    xlabel="Iteration", ylabel="Log-Likelihood",
    legend=false, lw=2, color=:navy, marker=:circle, ms=3,
    size=(700, 400));
display(p_conv);

## Emission Distributions

Each hidden state $k$ emits observations from a Gaussian $\mathcal{N}(\mu_k, \sigma_k)$.
Overlaying these on the observed histogram shows how the mixture of Gaussians decomposes
the marginal return distribution.

In [ ]:
# --- EMISSION PDFs OVERLAY ---
x_grid = range(-800, 800, length=2000);
colors = cgrad(:RdYlBu, K, categorical=true);

p_emit = plot(title="Learned Emission Distributions (K=$K)", titlefontsize=10,
    xlabel="Excess Growth Rate", ylabel="Density", legend=:topright, size=(900, 500));

# Observed histogram
histogram!(p_emit, R, normalize=true, bins=150, alpha=0.3, color=:gray, label="Observed");

# Overlay each state's Gaussian PDF
for s in 1:K
    d = model.emission[s];
    plot!(p_emit, x_grid, pdf.(d, x_grid), lw=2, color=colors[s],
        label="State $s (μ=$(round(mean(d),digits=1)), σ=$(round(std(d),digits=1)))", alpha=0.85);
end

xlims!(p_emit, -800, 800);
display(p_emit);

## Transition Matrix Heatmap

The $K \times K$ transition matrix $T_{ij} = P(s_{t+1} = j \mid s_t = i)$ encodes regime persistence
(diagonal) and switching behavior (off-diagonal). Displayed in $\log_{10}$ scale to reveal
rare transitions.

In [ ]:
# --- EXTRACT TRANSITION MATRIX ---
T_mat = zeros(K, K);
for i in 1:K
    T_mat[i, :] = probs(model.transition[i]);
end

# Log10 heatmap
T_log = log10.(T_mat .+ 1e-10);

p_trans = heatmap(T_log, title="Transition Matrix (log₁₀ scale, K=$K)", titlefontsize=10,
    xlabel="To State", ylabel="From State", color=:viridis,
    yflip=true, aspect_ratio=:equal, size=(600, 500));
display(p_trans);

## Stationary Distribution

The stationary distribution $\pi$ satisfies $\pi = \pi T$. We approximate it by matrix powering:
$\pi \approx e_1^\top T^{1000}$. This gives the long-run fraction of time spent in each regime.

In [ ]:
# --- STATIONARY DISTRIBUTION ---
π_stat = (T_mat^1000)[1, :];

p_pi = bar(1:K, π_stat, title="Stationary Distribution π (K=$K)", titlefontsize=10,
    xlabel="State", ylabel="Probability", legend=false,
    color=colors[1:K], alpha=0.8, size=(700, 400));
display(p_pi);

println("Sum of π: $(round(sum(π_stat), digits=6))");
for s in 1:K
    println("  State $s: π = $(round(π_stat[s], digits=4))");
end

## Residence Times

The expected natural residence time in state $k$ under the Markov chain is
$\tau_k = 1 / (1 - T_{kk})$ (in trading days). Short residence in tail states is a limitation
of the pure CHMM and motivates the jump mechanism explored in later notebooks.

In [ ]:
# --- RESIDENCE TIMES ---
residence_times = [1.0 / (1.0 - T_mat[k, k]) for k in 1:K];

p_res = bar(1:K, residence_times, title="Expected Residence Time per State (K=$K)", titlefontsize=10,
    xlabel="State", ylabel="Steps (trading days)", legend=false,
    color=colors[1:K], alpha=0.8, size=(700, 400));
display(p_res);

# Print table
println("State | T_kk      | E[τ_k] (days)");
println("------+-----------+--------------");
for k in 1:K
    println("  $(lpad(k,2))  | $(lpad(round(T_mat[k,k], digits=4),9)) | $(round(residence_times[k], digits=2))");
end

## Viterbi Decoding: Regime Overlay

The Viterbi algorithm finds the most likely hidden state sequence given the observations.
We overlay the decoded regimes on the in-sample close price series to visualize how the model
partitions the time series into distinct volatility regimes.

In [ ]:
# --- VITERBI DECODING ---
decoded_states = viterbi(R, model);

# Close prices (aligned with returns: skip first day)
close_prices = dataset[ticker].close[2:end];
dates = dataset[ticker].date[2:end];
T_obs = length(close_prices);

# Regime overlay plot
p_viterbi = plot(title="Viterbi Decoded Regimes on $(ticker) Close Prices", titlefontsize=10,
    xlabel="Trading Day Index", ylabel="Close Price (\$)", legend=:topleft, size=(1100, 500));

# Plot close price as thin gray line
plot!(p_viterbi, 1:T_obs, close_prices, lw=0.5, color=:gray, alpha=0.6, label="Close");

# Overlay colored scatter by regime
for s in 1:K
    idx = findall(decoded_states .== s);
    scatter!(p_viterbi, idx, close_prices[idx], ms=1.5, color=colors[s],
        label="State $s", alpha=0.7, markerstrokewidth=0);
end

display(p_viterbi);

## Simulate N_PATHS Paths

We draw `N_PATHS` independent Monte Carlo trajectories from the trained CHMM. Each path:
1. Samples a starting state from the stationary distribution $\pi$
2. Evolves the hidden state chain via the transition matrix (functor call: `model(s0, n_steps)`)
3. Generates continuous returns by sampling from the emission distribution: `rand(model.emission[state])`

In [ ]:
# --- SIMULATE N_PATHS ---
start_dist = Categorical(π_stat);
n_steps = length(R);
sim_archive = Array{Float64,2}(undef, n_steps, N_PATHS);

for i in 1:N_PATHS
    s0 = rand(start_dist);
    states = model(s0, n_steps);
    for j in 1:n_steps
        sim_archive[j, i] = rand(model.emission[states[j]]);
    end
end

println("CHMM: $(N_PATHS) paths of length $(n_steps) simulated.");

## Validation Metrics

We evaluate the simulated ensemble against the observed data using five complementary metrics:

| Metric | What it measures |
|--------|------------------|
| **KS pass rate** | Fraction of paths not rejected by the Kolmogorov-Smirnov test ($\alpha = 0.05$) |
| **Excess kurtosis** | Tail heaviness — the model should match the observed leptokurtosis |
| **ACF-MAE** | Mean absolute error of $\text{ACF}(|G_t|)$ — captures volatility clustering fidelity |
| **Wasserstein-1** | Earth-mover distance between observed and simulated quantiles |
| **Hellinger** | Histogram overlap distance ($0 =$ identical, $1 =$ disjoint) |

In [ ]:
# --- VALIDATION METRICS FUNCTION ---
function compute_validation_metrics(observed::Vector{Float64}, simulated_archive::Matrix{Float64};
                                    α=0.05, L=252)

    n_paths = size(simulated_archive, 2);
    n_obs = length(observed);

    # Observed targets
    μ_obs = mean(observed); σ_obs = std(observed);
    kurt_obs = sum(((observed .- μ_obs) ./ σ_obs).^4) / n_obs - 3.0;
    acf_obs = autocor(abs.(observed), 1:L);

    # Per-path accumulators
    ks_pass = 0;
    kurt_sum = 0.0; acf_mae_sum = 0.0;
    w1_sum = 0.0; hell_sum = 0.0;

    for i in 1:n_paths
        sim = simulated_archive[:, i];

        # KS test
        ks_pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        if ks_pval > α; ks_pass += 1; end

        # Excess kurtosis
        μ_s = mean(sim); σ_s = std(sim);
        kurt_sum += sum(((sim .- μ_s) ./ σ_s).^4) / length(sim) - 3.0;

        # ACF-MAE on absolute returns
        acf_sim = autocor(abs.(sim), 1:L);
        acf_mae_sum += mean(abs.(acf_obs .- acf_sim));

        # Wasserstein-1 (sorted quantile difference)
        obs_sorted = sort(observed);
        sim_sorted = sort(sim);
        n_min = min(length(obs_sorted), length(sim_sorted));
        obs_q = [obs_sorted[round(Int, k * length(obs_sorted) / n_min)] for k in 1:n_min];
        sim_q = [sim_sorted[round(Int, k * length(sim_sorted) / n_min)] for k in 1:n_min];
        w1_sum += mean(abs.(obs_q .- sim_q));

        # Hellinger distance (histogram-based)
        edges = range(minimum([observed; sim]) - 10, maximum([observed; sim]) + 10, length=101);
        h_obs = fit(Histogram, observed, edges).weights ./ length(observed);
        h_sim = fit(Histogram, sim, edges).weights ./ length(sim);
        hell_sum += sqrt(sum((sqrt.(h_obs) .- sqrt.(h_sim)).^2)) / sqrt(2);
    end

    # Build results
    ks_rate = round(100 * ks_pass / n_paths, digits=1);
    kurt_sim = round(kurt_sum / n_paths, digits=2);
    kurt_o = round(kurt_obs, digits=2);
    acf_mae = round(acf_mae_sum / n_paths, digits=4);
    w1 = round(w1_sum / n_paths, digits=3);
    hell = round(hell_sum / n_paths, digits=4);

    return (ks_rate=ks_rate, kurtosis_sim=kurt_sim, kurtosis_obs=kurt_o,
            acf_mae=acf_mae, wasserstein=w1, hellinger=hell);
end;

In [ ]:
# --- COMPUTE AND PRINT METRICS ---
metrics = compute_validation_metrics(R, sim_archive; L=L);

println("╔══════════════════════════════════════════════════════╗");
println("║  Validation: CHMM (K=$K, $ticker)                    ║");
println("╠══════════════════════════════════════════════════════╣");
println("║  KS pass rate (α=0.05):  $(lpad(metrics.ks_rate, 8))%              ║");
println("║  Excess kurtosis (sim):  $(lpad(metrics.kurtosis_sim, 8))               ║");
println("║  Excess kurtosis (obs):  $(lpad(metrics.kurtosis_obs, 8))               ║");
println("║  ACF-MAE |Gₜ|:          $(lpad(metrics.acf_mae, 8))               ║");
println("║  Wasserstein-1:          $(lpad(metrics.wasserstein, 8))               ║");
println("║  Hellinger:              $(lpad(metrics.hellinger, 8))               ║");
println("╚══════════════════════════════════════════════════════╝");

## Figure 3: Three-Panel In-Sample Comparison

- **(a)** Marginal density: histogram of observed returns with simulated kernel density overlay
- **(b)** ACF($|G_t|$): observed vs simulated mean with 10th--90th percentile band (volatility clustering)
- **(c)** Tail Q-Q plot: observed vs simulated quantiles from the 0.1st to 99.9th percentile

In [ ]:
# --- FIGURE 3a: MARGINAL DENSITY ---
p3a = plot(title="(a) Marginal Density (KS: $(metrics.ks_rate)%)",
    titlefontsize=9, xlabel="Excess Growth Rate", ylabel="Density");

histogram!(p3a, R, normalize=true, bins=150, alpha=0.3, color=:gray, label="Observed");
density!(p3a, vec(sim_archive), lw=2, color=:blue, alpha=0.7, label="CHMM (K=$K)");
xlims!(p3a, -800, 800);

# --- FIGURE 3b: ACF(|G|) COMPARISON ---
τ = 1:L;
acf_obs = autocor(abs.(R), τ);

n_acf_paths = min(200, N_PATHS);
acf_archive = hcat([autocor(abs.(sim_archive[:, i]), τ) for i in 1:n_acf_paths]...);
acf_mean = mean(acf_archive, dims=2)[:];
acf_p10 = [quantile(acf_archive[t, :], 0.10) for t in 1:L];
acf_p90 = [quantile(acf_archive[t, :], 0.90) for t in 1:L];

p3b = plot(τ, acf_obs, lw=2, color=:red, ls=:dash, label="Observed",
    title="(b) ACF(|Gₜ|) — Volatility Clustering", titlefontsize=9,
    xlabel="Lag (trading days)", ylabel="ACF(|Gₜ|)");
plot!(p3b, τ, acf_mean, lw=2, color=:blue, label="CHMM mean");
plot!(p3b, τ, acf_p10, fillrange=acf_p90, alpha=0.15, color=:blue, label="10-90th pctl");

# --- FIGURE 3c: TAIL Q-Q PLOT ---
probs_qq = range(0.001, 0.999, length=200);
q_obs = quantile(R, probs_qq);
q_sim = quantile(vec(sim_archive), probs_qq);

p3c = plot(q_obs, q_obs, lw=2, color=:black, ls=:dash, label="Perfect",
    title="(c) Tail Q-Q Plot (0.1st-99.9th pctl)", titlefontsize=9,
    xlabel="Observed Quantiles", ylabel="Simulated Quantiles");
scatter!(p3c, q_obs, q_sim, ms=3, alpha=0.6, color=:blue, label="CHMM (K=$K)");

# --- COMBINE ---
fig3 = plot(p3a, p3b, p3c, layout=(1, 3), size=(1400, 400),
    plot_title="Figure 3: In-Sample Comparison — $(ticker) (K=$K)",
    plot_titlefontsize=12);
display(fig3);

savefig(fig3, joinpath(_PATH_TO_FIGURES, "Fig-3-CHMM-$(ticker)-K$(K).svg"));
savefig(fig3, joinpath(_PATH_TO_FIGURES, "Fig-3-CHMM-$(ticker)-K$(K).pdf"));

## Disclaimer

This content is offered solely for research and educational purposes. It does not constitute
financial advice, investment recommendations, or trading strategies.